In [1]:
import time
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForCausalLM, AdamW, get_scheduler
from datasets import Dataset
from torch.cuda.amp import GradScaler, autocast

C:\Users\ibrahim\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [3]:
# Load JSON datasets
train_dataset = Dataset.from_json('train-open.json')  # Replace with actual path
val_dataset = Dataset.from_json('val-open.json')
test_dataset = Dataset.from_json('test-open.json')

Generating train split: 58676 examples [00:00, 175137.01 examples/s]
Generating train split: 12715 examples [00:00, 192028.63 examples/s]
Generating train split: 12592 examples [00:00, 171273.25 examples/s]


In [ ]:
# Load the tokenizer and set pad token
tokenizer = AutoTokenizer.from_pretrained("aubmindlab/aragpt2-medium")
tokenizer.pad_token = tokenizer.eos_token

# Load the model
model = AutoModelForCausalLM.from_pretrained("aubmindlab/aragpt2-medium")
model.to(device)

C:\Users\ibrahim\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\huggingface_hub\file_download.py:140: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ibrahim\.cache\huggingface\hub\models--aubmindlab--aragpt2-medium. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Error while downlo

In [6]:
# Tokenization function
def preprocess_function(examples):
    tokenized = tokenizer(
        examples["question"],  # Field for questions
        examples["answer"],    # Field for answers
        truncation=True,
        padding="max_length",  # Ensure uniform input length
        max_length=512         # Adjust based on model capability
    )
    # Set labels to be the same as input_ids for language modeling
    tokenized["labels"] = tokenized["input_ids"]
    return tokenized

# Tokenize datasets
train_dataset = train_dataset.map(preprocess_function, batched=True)
val_dataset = val_dataset.map(preprocess_function, batched=True)
test_dataset = test_dataset.map(preprocess_function, batched=True)

# **DEBUGGING STEP**
train_dataset = train_dataset.select(range(100))
val_dataset = val_dataset.select(range(50))
test_dataset = test_dataset.select(range(50))  

# Select necessary columns
columns_to_keep = ["input_ids", "attention_mask", "labels"]
train_dataset.set_format(type="torch", columns=columns_to_keep)
val_dataset.set_format(type="torch", columns=columns_to_keep)
test_dataset.set_format(type="torch", columns=columns_to_keep)

# Create DataLoaders
train_dataloader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=16)

Map:   0%|          | 0/12715 [00:00<?, ? examples/s]

In [7]:
# Define optimizer, scheduler, and scaler
optimizer = AdamW(model.parameters(), lr=5e-5)
num_training_steps = len(train_dataloader) * 3  # Assuming 3 epochs
scheduler = get_scheduler("linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=num_training_steps)
scaler = GradScaler()

C:\Users\Administrator\AppData\Roaming\Python\Python312\site-packages\transformers\optimization.py:591: FutureWarning: This implementation of AdamW is deprecated and will be removed in a future version. Use the PyTorch implementation torch.optim.AdamW instead, or set `no_deprecation_warning=True` to disable this warning
  warnings.warn(
C:\Users\Administrator\AppData\Local\Temp\ipykernel_13620\1579902327.py:5: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
C:\Users\Administrator\AppData\Roaming\Python\Python312\site-packages\torch\amp\grad_scaler.py:132: UserWarning: torch.cuda.amp.GradScaler is enabled, but CUDA is not available.  Disabling.
  warnings.warn(


In [8]:
# Training Loop
epochs = 3
for epoch in range(epochs):
    model.train()
    start_epoch = time.time()

    for step, batch in enumerate(train_dataloader):
        optimizer.zero_grad()

        # Move batch to device
        batch = {key: value.to(device) for key, value in batch.items()}

        start_batch = time.time()

        # Mixed precision forward and backward pass
        with autocast():
            outputs = model(input_ids=batch['input_ids'], labels=batch['labels'])
            loss = outputs.loss

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()

        # Log batch stats
        if step % 10 == 0:
            print(f"Epoch {epoch+1}, Step {step}, Loss: {loss.item():.4f}, Batch Time: {time.time() - start_batch:.2f}s")

    print(f"Epoch {epoch+1} completed in {time.time() - start_epoch:.2f}s")

    # Evaluation on validation set
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for val_batch in val_dataloader:
            val_batch = {key: value.to(device) for key, value in val_batch.items()}

            with autocast():
                val_outputs = model(input_ids=val_batch['input_ids'], labels=val_batch['labels'])
                total_loss += val_outputs.loss.item()

    avg_val_loss = total_loss / len(val_dataloader)
    print(f"Validation Loss after Epoch {epoch+1}: {avg_val_loss:.4f}")

print("Training complete!")

C:\Users\Administrator\AppData\Local\Temp\ipykernel_13620\2982656578.py:16: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
C:\Users\Administrator\AppData\Roaming\Python\Python312\site-packages\torch\amp\autocast_mode.py:266: UserWarning: User provided device_type of 'cuda', but CUDA is not available. Disabling
  warnings.warn(


: 

In [ ]:
# Save the model
model.save_pretrained("./optimized_finetuned_model")
tokenizer.save_pretrained("./optimized_finetuned_model")